In [8]:
import os
import json
import numpy as np
import pandas as pd

def find_model_files(root_folder):
    """Encontra pares de arquivos (h_matrix.txt, args.json) em uma pasta."""
    model_paths = []
    for dirpath, _, filenames in os.walk(root_folder):
        if 'h_matrix.txt' in filenames and 'args.json' in filenames:
            model_paths.append({
                "h_matrix": os.path.join(dirpath, 'h_matrix.txt'),
                "args": os.path.join(dirpath, 'args.json')
            })
    return model_paths

# Copiada diretamente da sua especificação para garantir consistência
def percept_preprocess(observation, num_percepts_list):
    """Pré-processa a observação para gerar um índice de percepção único."""
    percept = 0
    for which_feature in range(len(observation)):
        percept += int(observation[which_feature] * np.prod(num_percepts_list[:which_feature]))
    return percept

def get_prob_p(h_matrix, state_phi, timer_omega, colision_lambda, args):
    """Calcula p(ϕ, ω, λ) - a probabilidade de TROCAR de estado."""
    
    max_steps_per_trial = args['max_steps_per_episode'] // 20
    
    # Determina a estrutura da observação com base nos argumentos
    if args.get('colision', 0): # Usa .get para compatibilidade com JSONs antigos
        num_percepts_list = [2, max_steps_per_trial, 2]
        observation = [state_phi, timer_omega, colision_lambda]
    else:
        num_percepts_list = [2, max_steps_per_trial]
        observation = [state_phi, timer_omega]
        
    percept = percept_preprocess(observation, num_percepts_list)
    
    h_vector = h_matrix[:, percept]
    
    # Calcula a probabilidade baseada na política
    policy = args.get('policy', 'standard')
    if np.sum(h_vector) == 0: # Evita divisão por zero
        return 0.0

    if policy == 'standard':
        prob_distr = h_vector / np.sum(h_vector)
    elif policy == 'softmax':
        beta = args.get('beta_softmax', 1.0)
        h_vector_mod = beta * (h_vector - np.max(h_vector))
        prob_distr = np.exp(h_vector_mod) / np.sum(np.exp(h_vector_mod))
    else:
        raise ValueError(f"Política desconhecida: {policy}")

    # Retorna a probabilidade da AÇÃO 1 (trocar de estado)
    return prob_distr[1]

def calculate_average_duration(h_matrix, args, state_phi, colision_lambda):
    """Calcula ⟨ω⟩, o tempo médio de duração para um estado fixo."""
    
    max_steps = args.get('max_steps_per_episode', 200000) // 20
    dt = args.get('dt', 1.0)
    
    prob_p_array = np.array([get_prob_p(h_matrix, state_phi, omega, colision_lambda, args) for omega in range(max_steps)])
    
    prob_no_switch = 1 - prob_p_array
    
    # Previne underflow numérico em simulações muito longas
    with np.errstate(under='ignore'):
        prob_survived_until = np.cumprod(prob_no_switch)
    
    prob_survived_until = np.insert(prob_survived_until, 0, 1)[:-1]

    prob_exact_duration = prob_p_array * prob_survived_until
    
    time_steps = np.arange(max_steps) * dt
    avg_duration = np.sum(time_steps * prob_exact_duration)
    
    return avg_duration

In [9]:
os.listdir("../data/models/")

['.ipynb_checkpoints',
 'novo_colision_repulsive_k=1_R=0',
 'novo_colision_repulsive_k=0_R=0.005',
 'novo_periodic_k=1_R=0',
 'baseline_periodico',
 'novo_colision_repulsive_k=1_R=0.005',
 'novo_periodic_k=0_R=0',
 'baseline_repulsivo',
 'novo_colision_repulsive_k=0_R=0',
 'novo_colision_repulsive_k=2_R=0']

In [13]:

# --- CÉLULA DE EXECUÇÃO ---

# 1. ESPECIFIQUE O CAMINHO DA PASTA AQUI
folder_to_analyze = "../data/models/novo_colision_repulsive_k=1_R=0"

# 2. EXECUTE A ANÁLISE
if not os.path.isdir(folder_to_analyze):
    print(f"ERRO: A pasta especificada não foi encontrada: '{folder_to_analyze}'")
else:
    model_files = find_model_files(folder_to_analyze)
    
    if not model_files:
        print(f"Nenhum par de arquivos (h_matrix.txt, args.json) encontrado em '{folder_to_analyze}'.")
    else:
        print(f"Analisando {len(model_files)} modelos da pasta '{os.path.basename(folder_to_analyze)}'...")
        
        results = []
        # Carrega os argumentos do primeiro modelo para determinar a estrutura
        with open(model_files[0]['args'], 'r') as f:
            sample_args = json.load(f)
        
        # Para cada modelo encontrado...
        for model_path in model_files:
            h_matrix = np.loadtxt(model_path["h_matrix"], delimiter=',')
            with open(model_path['args'], 'r') as f:
                args = json.load(f)
            
            # Calcula ⟨ω⟩ para cada estado sem colisão
            avg_w_passivo = calculate_average_duration(h_matrix, args, state_phi=0, colision_lambda=0)
            avg_w_ativo = calculate_average_duration(h_matrix, args, state_phi=1, colision_lambda=0)
            
            result_row = {
                '⟨ω⟩ Passivo (s/ col)': avg_w_passivo,
                '⟨ω⟩ Ativo (s/ col)': avg_w_ativo,
            }
            
            # Se o modelo foi treinado com colisão, calcula os casos de colisão também
            if args.get('colision', 0):
                avg_w_passivo_col = calculate_average_duration(h_matrix, args, state_phi=0, colision_lambda=1)
                avg_w_ativo_col = calculate_average_duration(h_matrix, args, state_phi=1, colision_lambda=1)
                result_row['⟨ω⟩ Passivo (c/ col)'] = avg_w_passivo_col
                result_row['⟨ω⟩ Ativo (c/ col)'] = avg_w_ativo_col

            results.append(result_row)
        
        # Converte os resultados para um DataFrame do Pandas para fácil agregação
        df_results = pd.DataFrame(results)
        
        # Calcula a média e o desvio padrão para cada coluna
        summary_stats = df_results.agg(['mean', 'std'])
        
        print("\n--- Análise da Duração Média dos Estados ⟨ω⟩ ---")
        print("Valores calculados para cada um dos {} modelos:".format(len(df_results)))
        # Configura o Pandas para exibir os floats com 2 casas decimais
        pd.set_option('display.float_format', '{:.2f}'.format)
        print(df_results.to_string()) # .to_string() garante que todas as colunas sejam exibidas
        
        print("\n--- Estatísticas Agregadas ---")
        print(summary_stats.to_string())
        
        print("\n--- Interpretação da Estratégia Média ---")
        mean_passivo_sc = summary_stats.loc['mean', '⟨ω⟩ Passivo (s/ col)']
        mean_ativo_sc = summary_stats.loc['mean', '⟨ω⟩ Ativo (s/ col)']
        print(f"- Longe da parede, a bactéria busca (Passivo) por em média {mean_passivo_sc:.2f}s antes de desistir.")
        print(f"- Longe da parede, a bactéria explora (Ativo) por em média {mean_ativo_sc:.2f}s antes de decidir procurar.")
        
        if '⟨ω⟩ Passivo (c/ col)' in summary_stats.columns:
            mean_passivo_cc = summary_stats.loc['mean', '⟨ω⟩ Passivo (c/ col)']
            mean_ativo_cc = summary_stats.loc['mean', '⟨ω⟩ Ativo (c/ col)']
            print(f"- Em colisão, a bactéria busca (Passivo) por em média {mean_passivo_cc:.2f}s.")
            print(f"- Em colisão, a bactéria explora (Ativo) por em média {mean_ativo_cc:.2f}s.")
            if mean_ativo_cc < mean_ativo_sc:
                print("  ↳ O tempo de exploração em colisão é menor, sugerindo que o agente aprendeu que é ineficiente insistir no modo ativo ao bater na parede.")

Analisando 4 modelos da pasta 'novo_colision_repulsive_k=1_R=0'...

--- Análise da Duração Média dos Estados ⟨ω⟩ ---
Valores calculados para cada um dos 4 modelos:
   ⟨ω⟩ Passivo (s/ col)  ⟨ω⟩ Ativo (s/ col)  ⟨ω⟩ Passivo (c/ col)  ⟨ω⟩ Ativo (c/ col)
0                129.50              129.97                103.88              308.25
1                127.32              164.22                107.40              320.22
2                103.75              103.85                 99.32               78.07
3                125.58              749.50                 99.53              395.96

--- Estatísticas Agregadas ---
      ⟨ω⟩ Passivo (s/ col)  ⟨ω⟩ Ativo (s/ col)  ⟨ω⟩ Passivo (c/ col)  ⟨ω⟩ Ativo (c/ col)
mean                121.54              286.89                102.53              275.63
std                  11.97              309.40                  3.87              137.31

--- Interpretação da Estratégia Média ---
- Longe da parede, a bactéria busca (Passivo) por em média 121.5